In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch master https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile inference.py inference/*.py encode/*.py inference/models/*.py
!cd /kaggle/working/Real-ESRGAN && python inference.py --help >/dev/null
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "(libx265|hevc_nvenc|libsvtav1|libaom-av1|av1_nvenc)" || true


In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/bz.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "24000/1001"

START_TIME = 0
TEST_SECONDS = 10

GPU_IDS = "0,1"

# 源质量档：A=关闭 BasicVSR++；B=轻度压缩/噪声；C=明显压缩/噪声。
# Real-ESRGAN 在所有档位下都保持 full-frame；BasicVSR++ 仅作为同分辨率前置修复。
SOURCE_PROFILE = "A"

# 推理固定为 full-frame：CUDA 自动使用 FP16 + channels_last；输入保持源分辨率。

# 编码器：
#   CPU HEVC = "libx265"
#   GPU HEVC = "hevc_nvenc"
#   CPU AV1  = "libsvtav1"  # 缺失时自动尝试 libaom-av1
#   CPU AV1  = "libaom-av1"
#   GPU AV1  = "av1_nvenc"
VIDEO_CODEC = "hevc_nvenc"
CRF = 18
PRESET = "medium"
SVTAV1_PRESET = 6
AOM_CPU_USED = 6
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0

AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [ ]:
import subprocess
import sys

command = [
    sys.executable, "/kaggle/working/Real-ESRGAN/inference.py",
    "--input", INPUT_VIDEO,
    "--output", OUTPUT_VIDEO,
    "--model", MODEL,
    "--model-path", MODEL_PATH,
    "--scale", str(SCALE),
    "--fps", str(FPS),
    "--gpu-ids", GPU_IDS,
    "--source-profile", SOURCE_PROFILE,
    "--video-codec", VIDEO_CODEC,
    "--crf", str(CRF),
    "--preset", PRESET,
    "--svtav1-preset", str(SVTAV1_PRESET),
    "--aom-cpu-used", str(AOM_CPU_USED),
    "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET,
    "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC,
    "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME),
    "--test-seconds", str(TEST_SECONDS),
    "--ffmpeg-bin", "ffmpeg",
    "--ffprobe-bin", "ffprobe",
]

_result = subprocess.run(command, check=True)
